# 08a · Exportar los pesos del DeepFM — correr en Colab

Etapa 8, paso a. Reentrena el DeepFM con la configuración elegida en el paso 7c (semilla 0) y guarda sus pesos en `deepfm_pesos.npz`. Con ese archivo, `recomendar()` (paso b) predice en local con numpy, sin PyTorch. También guarda las predicciones de este modelo sobre los candidatos de test para comprobar en local que la versión numpy da lo mismo.

**Cómo correrlo:** Colab → *Ejecutar todas* (no hace falta GPU; tarda 1–2 min). Se descarga `etapa8a_resultados.zip`: pégalo **sin descomprimir** en la carpeta `modelo-fly`, al lado de `README.md`.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

RAMA = "hito3/etapa8-bd-comun"
URL = "https://github.com/sowiexsker894/Predicci-n-de-compatibilidad-de-outfits-desde-el-conectoma-del-Giant-Fiber-System.git"
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not Path("/content/modelo-fly").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, URL, "/content/modelo-fly"], check=True)
    ROOT = Path("/content/modelo-fly")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

from modelo_fly.caracteristicas import GRUPOS_SLOT, codificar, codificar_usuarios
from modelo_fly.deepfm import DeepFM, entrenar, predecir
from modelo_fly.usuarios import cargar_items

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
PROC = ROOT / "data" / "processed" / "hito3"
CFG = json.loads((ROOT / "results" / "hito3" / "fase1" / "deepfm_meta.json").read_text())["configuracion"]
print("Configuración del paso 7c:", CFG, "| dispositivo:", DISPOSITIVO)

In [ ]:
items, atrib, _ = cargar_items(PROC)
usuarios = pd.read_csv(PROC / "usuarios.csv")
historial = pd.read_csv(PROC / "historial.csv")
cand = pd.read_csv(PROC / "candidatos.csv")
fila_u, N_VOCAB = codificar_usuarios(usuarios)

def preparar(df):
    return np.array([fila_u[u] for u in df["user_id"]]), codificar(df["item_ids"].str.split("|").tolist(), atrib)

datos = {p: preparar(historial[historial["particion"] == p]) for p in ["train", "valid", "test"]}
y = {p: historial.loc[historial["particion"] == p, "aceptado"].to_numpy() for p in datos}
m = DeepFM(N_VOCAB, datos["train"][1].shape[1], list(GRUPOS_SLOT.values()), k=CFG["k"], dropout=CFG["dropout"])
t = time.time()
curva = entrenar(m, *datos["train"], y["train"], *datos["valid"], y["valid"], semilla=0, peso=CFG["peso"],
                 dispositivo=DISPOSITIVO)
print(f"{len(curva)} épocas, pérdida valid {min(curva):.4f}, {time.time() - t:.0f} s | AUC historial test:",
      round(roc_auc_score(y["test"], predecir(m, *datos["test"], DISPOSITIVO)), 3))

In [ ]:
SALIDA = ROOT / "salida_8a"
SALIDA.mkdir(exist_ok=True)
pesos = {k: v.detach().cpu().numpy() for k, v in m.state_dict().items()}
np.savez_compressed(SALIDA / "deepfm_pesos.npz", **pesos)
te = cand[cand["particion"] == "test"].reset_index(drop=True)
U_te, X_te = preparar(te)
te[["user_id", "outfit_id"]].assign(p_torch=predecir(m, U_te, X_te, DISPOSITIVO)).to_csv(
    SALIDA / "deepfm_verificacion.csv.gz", index=False)
(SALIDA / "deepfm_pesos_meta.json").write_text(json.dumps({
    "configuracion": CFG, "semilla": 0, "epocas": len(curva), "perdida_valid": min(curva),
    "torch": torch.__version__, "capas": {k: list(v.shape) for k, v in pesos.items()}}, indent=2))
import shutil
zip_path = shutil.make_archive(str(ROOT / "etapa8a_resultados"), "zip", SALIDA)
print("Listo:", zip_path, sorted(p.name for p in SALIDA.iterdir()))
if EN_COLAB:
    from google.colab import files
    files.download(zip_path)